<a href="https://colab.research.google.com/github/simecek/dspracticum2026/blob/main/lesson05/04_bert_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 05 · Notebook 4: BERT, and training a classifier

GPT reads a text from left to right and predicts the next token. That is what you need for **writing** text. But many tasks are about **understanding** a text you already have: is this review positive? Is this e-mail spam? Which words are names of people? For these, the go-to model is **BERT** (Google, 2018): the same transformer block, but every token sees the **whole** text, to the left and to the right.

In this notebook we train a BERT-type model to tell positive Czech movie reviews from negative ones: the ČSFD reviews from notebook 3.

**Contents:** 0. Setup · 1. GPT vs BERT: a different mask · 2. Fill in the blank · 3. ModernBERT and mmBERT · 4. Training a classifier · 5. Using the classifier · 6. Summary

**Before you start:** *Runtime → Change runtime type → T4 GPU* (for section 4).

---
## 0. Setup

In [ ]:
import matplotlib.pyplot as plt
import torch
from transformers import pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

def show_predictions(fill, sentence, k=5):
    # Print the k most likely words for the [MASK] in the sentence.
    sentence = sentence.replace("[MASK]", fill.tokenizer.mask_token)    # every model has its own mask token
    print(sentence)
    for r in fill(sentence, top_k=k):
        print(f"   {r['score']:6.1%}  {r['token_str']!r}")

---
## 1. GPT vs BERT: a different mask

Remember the **causal mask** from last week: in GPT, a token may only look at itself and the tokens **before** it, because it must predict what comes next.

BERT has **no causal mask**: every token looks at every other token. That is the main difference inside the transformer block. GPT is a **decoder** (it generates text), BERT is an **encoder** (it turns a text into vectors).

In [ ]:
tokens = ["The", "dog", "barked", "at", "the", "mailman"]
T = len(tokens)
masks = {"GPT (causal mask)": torch.tril(torch.ones(T, T)), "BERT (no mask)": torch.ones(T, T)}

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, (title, mask) in zip(axes, masks.items()):
    ax.imshow(mask, cmap="Blues", vmin=0, vmax=1.5)
    ax.set_xticks(range(T), tokens, rotation=90)
    ax.set_yticks(range(T), tokens)
    ax.set_xlabel("... can look at this token")
    ax.set_title(title)
axes[0].set_ylabel("this token ...")
plt.tight_layout()
plt.show()

Take the sentence *"The ___ barked at the mailman."* GPT, reading from the left, sees only *"The"* when it has to fill the blank. BERT also sees *"barked"* and *"mailman"*. The price: BERT cannot generate text token by token. It is not a chatbot.

Otherwise the two are almost twins:

| | GPT-2 small | BERT-base |
|---|---|---|
| year | 2019 | 2018 |
| type | decoder: predicts the next token | encoder: fills in masked tokens |
| attention mask | causal | none, everyone sees everyone |
| blocks × vector size × heads | 12 × 768 × 12 | 12 × 768 × 12 |
| vocabulary | 50,257 (BPE) | 30,522 (WordPiece) |
| context | 1,024 tokens | 512 tokens |
| parameters | 124 million | 110 million |

---
## 2. Fill in the blank

How do you train a model that sees the whole text? "Predict the next token" would be cheating: the next token is right there. Instead, BERT is trained on **masked language modelling**: take a text, hide 15% of its tokens behind a special `[MASK]` token, and let the model guess them. Like GPT, it needs no labels, just a lot of text: BERT read all of English Wikipedia and 11,000 books.

The `fill-mask` pipeline lets us play this game with the trained model:

In [ ]:
bert_fill = pipeline("fill-mask", model="google-bert/bert-base-uncased", device=device)

show_predictions(bert_fill, "Paris is the [MASK] of France.")
show_predictions(bert_fill, "The [MASK] barked at the mailman.")

*(The table about `UNEXPECTED` keys: BERT was also trained on a second task, and the weights for it are in the file, but `fill-mask` does not need them. You can ignore it.)*

BERT is not even sure whether it was a man or a dog who barked.

**Try it:** Your own sentences. And this one: what did BERT learn about people from its training texts?

In [ ]:
show_predictions(bert_fill, "The nurse said that [MASK] would be late.")
show_predictions(bert_fill, "The doctor said that [MASK] would be late.")

---
## 3. ModernBERT and mmBERT

BERT is from 2018. In December 2024, [ModernBERT](https://huggingface.co/blog/modernbert) took the BERT recipe and updated it with what people had learned from building LLMs:

| | BERT-base (2018) | ModernBERT-base (2024) |
|---|---|---|
| training data | 3.3 billion words: Wikipedia and books | 2 trillion tokens: web, code, science |
| context | 512 tokens | 8,192 tokens |
| positions | learned table (like GPT-2) | RoPE: rotates queries and keys by an angle that depends on the position |
| attention | every block: all tokens | every 3rd block: all tokens, the others only the 128 nearest tokens (faster) |
| blocks, parameters | 12, 110 million | 22, 150 million |

In [ ]:
modern_fill = pipeline("fill-mask", model="answerdotai/ModernBERT-base", device=device)

show_predictions(modern_fill, "The [MASK] barked at the mailman.")
show_predictions(modern_fill, "def add(a, b):\n    return a [MASK] b")

The dog is back, and thanks to the code in its training data, ModernBERT also reads Python.

ModernBERT read only English. In 2025 came [mmBERT](https://huggingface.co/blog/mmbert): the same architecture, trained on 3 trillion tokens in more than 1,800 languages, **including Czech**. We will fine-tune its small version, `mmBERT-small` (140 million parameters):

In [ ]:
mm_fill = pipeline("fill-mask", model="jhu-clsp/mmBERT-small", device=device)

show_predictions(mm_fill, "Praha je hlavní [MASK] České republiky.")

---
## 4. Training a classifier

Now we teach mmBERT a new task: does a ČSFD review like the movie or not? We take the reviews with 0-1 stars as **negative** and 4-5 stars as **positive**, and drop the lukewarm ones.

This is **fine-tuning**, just as with the image models in lessons 2 and 3: a pre-trained network, plus a new small layer (the *head*) that we train on labelled examples. BERT gives one vector per token. The head takes their **average**, one vector for the whole review, and turns it into 2 scores: negative and positive.

First the data, with `datasets` (notebook 3):

In [ ]:
from datasets import load_dataset

reviews = load_dataset("fewshot-goes-multilingual/cs_csfd-movie-reviews")

def positive_or_negative(split, n):
    ds = reviews[split].filter(lambda row: row["rating_int"] in (0, 1, 4, 5))
    ds = ds.map(lambda row: {"label": int(row["rating_int"] >= 4)})     # 1 = positive, 0 = negative
    return ds.shuffle(seed=42).select(range(n))

train_ds = positive_or_negative("train", 2000)
test_ds = positive_or_negative("test", 500)
print(train_ds)
print("positive in training data:", sum(train_ds["label"]), "of", len(train_ds))

Then we tokenize all texts with `map(batched=True)`, as in notebook 3. Reviews longer than 256 tokens are cut off (`truncation`).

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "jhu-clsp/mmBERT-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["comment"], truncation=True, max_length=256)

train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)
print(train_ds[0]["input_ids"][:20])

The model: `AutoModelForSequenceClassification` loads mmBERT and adds a new classification head with 2 outputs:

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=2, id2label={0: "negative", 1: "positive"}, label2id={"negative": 0, "positive": 1})
print(model.classifier)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

The load report says it: the weights for word prediction (`decoder`) are `UNEXPECTED`, because we do not need them now, and the `classifier` is `MISSING`: it is new, with random weights.

We could write the training loop ourselves, as last week. But `transformers` has one ready, the **`Trainer`**. It does the same 5 steps, plus batching, padding, evaluation, logging and the GPU:

In [ ]:
from transformers import Trainer, TrainingArguments

def accuracy(eval_pred):
    logits, labels = eval_pred
    return {"accuracy": (logits.argmax(-1) == labels).mean()}

args = TrainingArguments(
    output_dir="csfd-classifier",
    num_train_epochs=2,
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",              # measure the accuracy on the test reviews after every epoch
    logging_steps=25,
    save_strategy="no",                 # do not save checkpoints during training
    fp16=torch.cuda.is_available(),     # faster on the GPU
    report_to="none",
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=test_ds,
                  processing_class=tokenizer, compute_metrics=accuracy)
trainer.train()

The table shows the training loss going down and, after every epoch, the accuracy on the 500 test reviews that the model has not seen during training.

---
## 5. Using the classifier

The trained model goes into a `pipeline`, like any model from the Hub:

In [ ]:
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device=device)

my_reviews = [
    "Naprosto skvělý film, herci výborní, doporučuji!",
    "Nuda, nuda, šeď. Škoda času.",
    "Ze začátku jsem se nudil, ale konec to zachránil.",
]
for review, result in zip(my_reviews, classifier(my_reviews)):
    print(f"{result['label']:8} {result['score']:.0%}   {review}")

**Compare with notebook 2**, where `gpt-5.6-luna` rated reviews without any training: here we needed 2,000 labelled reviews and a GPU for a few minutes. In return, our model is small, runs on our own computer, and costs nothing per review.

**Upload it to the Hub** (optional, about 560 MB), the same way as in notebook 1. Then anyone can use it with `pipeline("text-classification", model="your-username/csfd-sentiment")`:

In [ ]:
from huggingface_hub import whoami

user = whoami()["name"]
model.push_to_hub(f"{user}/csfd-sentiment")
tokenizer.push_to_hub(f"{user}/csfd-sentiment")
print(f"https://huggingface.co/{user}/csfd-sentiment")

**Try it:**
- Write your own reviews. Can you fool the classifier, e.g. with irony?
- Train with 200 or 500 reviews instead of 2,000. How much worse is the accuracy?
- Use all 6 classes (0-5 stars, `num_labels=6`, `label = rating_int`, and leave out the `filter`). How often does it guess the exact number of stars?
- Train `answerdotai/ModernBERT-base` on the same data. It does not know Czech: how much worse is it?

---
## 6. Summary

| | GPT | BERT |
|---|---|---|
| attention | causal: only the tokens before | all tokens |
| trained on | predicting the next token | filling in masked tokens |
| good for | generating text | understanding text: classification, search, finding names |
| pipeline | `text-generation` | `fill-mask`, `text-classification` |
| model class | `AutoModelForCausalLM` | `AutoModelForMaskedLM`, `AutoModelForSequenceClassification` |

Fine-tuning a classifier: `load_dataset` → `map(tokenize, batched=True)` → `AutoModelForSequenceClassification.from_pretrained(..., num_labels=...)` → `Trainer(...).train()` → `pipeline("text-classification", ...)`.

**New words:** encoder, decoder, masked language modelling, `[MASK]`, WordPiece, RoPE, classification head, fine-tuning, `Trainer`, epoch, accuracy.

### Further reading
- [BERT paper](https://arxiv.org/abs/1810.04805), Jay Alammar: [The Illustrated BERT](https://jalammar.github.io/illustrated-bert/)
- [Finally, a replacement for BERT](https://huggingface.co/blog/modernbert): the ModernBERT blog post; [mmBERT](https://huggingface.co/blog/mmbert): the multilingual version
- Hugging Face LLM course, [chapter 3: fine-tuning a pretrained model](https://huggingface.co/learn/llm-course/chapter3/1)